# Stage 2 HP tuning (scenarios A & B)

Tunes T5 wide RF/XGB + MLP on conservative non-test pools; assembles scenario **C** from 5.2 exports.


In [1]:
from pathlib import Path
import json
import pandas as pd

from utils.nn_stage2_data import load_nn_stage2_data
from utils.stage2_export import export_stage2_data
from utils.stage2_hp import (
    resolve_torch_device,
    tune_scenario_ab,
    write_scenario_hp_json,
)
from utils.benchmark_metrics import STAGE2_BEST_HP_DIR, STAGE2_DATA_DIR

HP_DIR = STAGE2_BEST_HP_DIR
HP_DIR.mkdir(parents=True, exist_ok=True)
device = resolve_torch_device()
print("device:", device)

device: mps


In [2]:
# Idempotent export snapshot for HP notebook
if not (STAGE2_DATA_DIR / "manifest.json").is_file():
    export_stage2_data(out_dir=STAGE2_DATA_DIR)
else:
    print("stage2_data export present; skip")

stage2_data export present; skip


In [3]:
data = load_nn_stage2_data()
trials = []
for scen in ("A", "B"):
    payload, mlp_log = tune_scenario_ab(data, scen, device=device, verbose=True)
    write_scenario_hp_json(payload, HP_DIR / f"{scen}.json")
    trials.append(mlp_log)
pd.concat(trials, ignore_index=True).to_parquet(
    HP_DIR / "hp_search_log.parquet", index=False
)

Scenario A: wide=283 rows (45 animals), long=2811 rows
  MLP trial 1/72 rmse_af=11.2333
  MLP trial 2/72 rmse_af=11.2336
  MLP trial 3/72 rmse_af=3.0703
  MLP trial 4/72 rmse_af=3.0704
  MLP trial 5/72 rmse_af=2.3789
  MLP trial 6/72 rmse_af=2.3181
  MLP trial 7/72 rmse_af=11.2487
  MLP trial 8/72 rmse_af=11.2489
  MLP trial 9/72 rmse_af=3.0863
  MLP trial 10/72 rmse_af=3.0860
  MLP trial 11/72 rmse_af=2.3250
  MLP trial 12/72 rmse_af=2.3170
  MLP trial 13/72 rmse_af=11.2659
  MLP trial 14/72 rmse_af=11.2661
  MLP trial 15/72 rmse_af=3.0880
  MLP trial 16/72 rmse_af=3.0886
  MLP trial 17/72 rmse_af=2.3365
  MLP trial 18/72 rmse_af=2.3353
  MLP trial 19/72 rmse_af=3.8486
  MLP trial 20/72 rmse_af=3.8486
  MLP trial 21/72 rmse_af=2.1669
  MLP trial 22/72 rmse_af=2.1637
  MLP trial 23/72 rmse_af=1.9011
  MLP trial 24/72 rmse_af=1.9060
  MLP trial 25/72 rmse_af=3.8574
  MLP trial 26/72 rmse_af=3.8574
  MLP trial 27/72 rmse_af=2.2747
  MLP trial 28/72 rmse_af=2.2748
  MLP trial 29/72 rmse_a

In [4]:
# Scenario C: merge T5 sklearn + Colab MLP (run scripts/export_liberman_t5_hp.py + merge if missing)
import subprocess

t5 = HP_DIR / "liberman_t5_sklearn.json"
if not t5.is_file():
    subprocess.run(["python", "scripts/export_liberman_t5_hp.py"], check=True, cwd=".")
c_out = HP_DIR / "C.json"
if not c_out.is_file():
    subprocess.run(["python", "scripts/merge_liberman_hp.py"], check=True, cwd=".")
assert c_out.is_file(), "C.json missing"
print("C.json OK")

C.json OK
